# **Import Library**

In [1]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import randint, uniform, loguniform
import optuna

# **Load Data**

In [4]:
data = pd.read_csv("/kaggle/input/datafixxxx/dataset_training_final_FIXED.csv")

# **Split**

In [7]:
X = data.drop(columns=['label', 'text', 'query', 'len_q', 'len_t'])
y = data['label']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# **Logistic Regression**

In [7]:
def objective_lr(trial):
    solver = trial.suggest_categorical('solver', ['liblinear', 'newton-cg', 'lbfgs', 'sag', 'saga'])
    C = trial.suggest_float('C', 0.001, 100, log=True)

    modellr = LogisticRegression(solver=solver, C=C, random_state=42)
    score = cross_val_score(
        modellr,
        X_train,
        y_train,
        cv=3,
        scoring='roc_auc'
    ).mean() 

    return score

In [8]:
studylr = optuna.create_study(direction='maximize')
studylr.optimize(objective_lr, n_trials=100)

[I 2025-12-11 07:37:58,099] A new study created in memory with name: no-name-8132bffe-1043-4c79-b126-f6a4fff4f146
[I 2025-12-11 07:37:58,559] Trial 0 finished with value: 0.7807632414635521 and parameters: {'solver': 'liblinear', 'C': 14.880415715079845}. Best is trial 0 with value: 0.7807632414635521.
[I 2025-12-11 07:38:00,268] Trial 1 finished with value: 0.7807065535482002 and parameters: {'solver': 'lbfgs', 'C': 0.5824121702215915}. Best is trial 0 with value: 0.7807632414635521.
[I 2025-12-11 07:38:00,740] Trial 2 finished with value: 0.7806708352024812 and parameters: {'solver': 'liblinear', 'C': 0.47311792762623517}. Best is trial 0 with value: 0.7807632414635521.
[I 2025-12-11 07:38:02,219] Trial 3 finished with value: 0.7801646582181684 and parameters: {'solver': 'lbfgs', 'C': 0.10003547093325828}. Best is trial 0 with value: 0.7807632414635521.
[I 2025-12-11 07:38:03,865] Trial 4 finished with value: 0.7336710061088748 and parameters: {'solver': 'saga', 'C': 0.00116835356934

In [9]:
bestlr = studylr.best_trial
print(f"parameter terbaik: {bestlr.params}")

parameter terbaik: {'solver': 'lbfgs', 'C': 99.54126403629274}


In [10]:
parameterlr=bestlr.params
lr = LogisticRegression(solver=parameterlr['solver'], C=parameterlr['C'], random_state=42)
lr.fit(X_train, y_train)

LogisticRegression(C=99.54126403629274, random_state=42)

In [11]:
y_scoreslr = lr.predict_proba(X_test)[:, 1]
map_scorelr = average_precision_score(y_test, y_scoreslr)
auc_roclr = roc_auc_score(y_test, y_scoreslr)

print("hasil LR")
print(f"MAP : {map_scorelr}")
print(f"AUC-ROC : {auc_roclr}")

hasil LR
MAP : 0.6286049537266903
AUC-ROC : 0.7760990752884399


In [12]:
joblib.dump(lr, "logisticregression.pkl")

['logisticregression.pkl']

# **SVM**

In [14]:
def objective_svm(trial):
    C = trial.suggest_float('C', 0.01, 10, log=True)
    gamma = trial.suggest_float('gamma', 0.01, 10, log=True)

    modelsvm = SVC(kernel='rbf', C=C, gamma=gamma)

    score = cross_val_score(
        modelsvm,
        X_train,
        y_train,
        cv=3,
        scoring='roc_auc'
    ).mean()

    return score
    

In [ ]:
studysvm = optuna.create_study(direction='maximize')
studysvm.optimize(objective_svm, n_trials=10)

[I 2025-12-08 01:15:05,893] A new study created in memory with name: no-name-82f8471c-dd1a-40e9-b32f-37f520e131a8
[I 2025-12-08 01:41:14,678] Trial 0 finished with value: 0.721885117408199 and parameters: {'C': 0.9981041812215098, 'gamma': 1.0062686398473342}. Best is trial 0 with value: 0.721885117408199.
[I 2025-12-08 02:10:53,097] Trial 1 finished with value: 0.7251831579742323 and parameters: {'C': 5.36831490456946, 'gamma': 0.3307013996684513}. Best is trial 1 with value: 0.7251831579742323.
[I 2025-12-08 02:38:51,157] Trial 2 finished with value: 0.7230218669193146 and parameters: {'C': 1.6073025004165198, 'gamma': 0.7755377649294619}. Best is trial 1 with value: 0.7251831579742323.
[I 2025-12-08 02:58:07,625] Trial 3 finished with value: 0.781684200610346 and parameters: {'C': 6.022672643176584, 'gamma': 0.05592036054407158}. Best is trial 3 with value: 0.781684200610346.
[I 2025-12-08 03:22:19,726] Trial 4 finished with value: 0.783793209665281 and parameters: {'C': 0.150081175

In [ ]:
bestsvm = studysvm.best_trial
print(f"parameter terbaik: {bestsvm.params}")

In [ ]:
parametersvm=bestsvm.params
svm = SVC(kernel='rbf', C=parametersvm['C'], gamma=parametersvm['gamma'])
svm.fit(X_train, y_train)

In [ ]:
y_scoressvm = svm.predict_proba(X_test)[:, 1]
map_scoresvm = average_precision_score(y_test, y_scoressvm)
auc_rocsvm = roc_auc_score(y_test, y_scoressvm)

print("hasil SVM")
print(f"MAP : {map_scoresvm}")
print(f"AUC-ROC : {auc_rocsvm}")

In [ ]:
joblib.dump(svm, "svm.pkl")

In [11]:
terbaiksvm = SVC(kernel='rbf', C=3.42, gamma=0.01, probability=True)
terbaiksvm.fit(X_train, y_train)

SVC(C=3.42, gamma=0.01, probability=True)

In [13]:
joblib.dump(terbaiksvm, "svm.pkl")

['svm.pkl']

In [ ]:
print()

In [ ]:
import joblib
model = joblib.load("svm.pkl")
print(model)